In [3]:
import gc
import pickle
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# VAE embeddings + CatBoost

## Load artifacts

In [4]:
DATA_DIR = '/kaggle/input/datasets/f0rzah/artifacts4/'
OUTPUT_DIR = '/kaggle/working/'

train_final = pd.read_parquet(f'{DATA_DIR}train_final.parquet')
items_features = pd.read_parquet(f'{DATA_DIR}items_features.parquet')
users_features = pd.read_parquet(f'{DATA_DIR}users_features.parquet')

with open(f'{DATA_DIR}evaluation_samples.pkl', 'rb') as f:
    eval_data = pickle.load(f)

print(f'train_final: {train_final.shape}')

train_final: (9178632, 25)


## Vocabularies

In [5]:
def build_vocab(series):
    vals = series.fillna('Unknown').astype(str).unique().tolist()
    if 'Unknown' in vals:
        vals.remove('Unknown')
    return {v: i for i, v in enumerate(['Unknown'] + vals)}

vocabularies = {
    'age': build_vocab(users_features['age']),
    'sex': build_vocab(users_features['sex']),
    'income': build_vocab(users_features['income']),
    'content_type': build_vocab(items_features['content_type']),
    'release_decade': build_vocab(items_features['release_decade']),
}

vocab_sizes = {k: len(v) for k, v in vocabularies.items()}
print(vocab_sizes)

{'age': 7, 'sex': 3, 'income': 7, 'content_type': 3, 'release_decade': 7}


## VAE dataset

In [6]:
train_positives = train_final[train_final['target'] == 1].reset_index(drop=True)
print(f'train_positives: {train_positives.shape}')

train_positives: (3059544, 25)


In [7]:
class VAEDataset(Dataset):
    def __init__(self, df, vocabularies):
        self.user_ids = df['user_id'].values.astype(np.int64)
        self.item_ids = df['item_id'].values.astype(np.int64)

        self.u_age = df['age'].fillna('Unknown').astype(str).map(vocabularies['age']).fillna(0).values.astype(np.int64)
        self.u_sex = df['sex'].fillna('Unknown').astype(str).map(vocabularies['sex']).fillna(0).values.astype(np.int64)
        self.u_income = df['income'].fillna('Unknown').astype(str).map(vocabularies['income']).fillna(0).values.astype(np.int64)
        self.u_kids = df['kids_flg'].fillna(0).values.astype(np.float32)

        self.i_type = df['content_type'].fillna('Unknown').astype(str).map(vocabularies['content_type']).fillna(0).values.astype(np.int64)
        self.i_decade = df['release_decade'].fillna('Unknown').astype(str).map(vocabularies['release_decade']).fillna(0).values.astype(np.int64)
        self.i_age_rating = df['age_rating_ord'].fillna(2).values.astype(np.float32)

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return {
            'u_age': self.u_age[idx],
            'u_sex': self.u_sex[idx],
            'u_income': self.u_income[idx],
            'u_kids': self.u_kids[idx],
            'i_type': self.i_type[idx],
            'i_decade': self.i_decade[idx],
            'i_age_rating': self.i_age_rating[idx],
        }

In [8]:
BATCH_SIZE = 512

train_dataset = VAEDataset(train_positives, vocabularies)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f'batches per epoch: {len(train_loader)}')

batches per epoch: 5975


## Architecture

In [9]:
class DenoisingWrapper:
    def __init__(self, noise_prob=0.15):
        self.noise_prob = noise_prob

    def add_noise(self, x, vocab_size):
        mask = torch.rand_like(x.float()) < self.noise_prob
        rand = torch.randint(0, vocab_size, x.shape, device=x.device)
        return torch.where(mask, rand, x)

In [10]:
class UserVAEEncoder(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=64, latent_dim=8):
        super().__init__()
        self.emb_age = nn.Embedding(vocab_sizes['age'], emb_dim)
        self.emb_sex = nn.Embedding(vocab_sizes['sex'], emb_dim // 2)
        self.emb_income = nn.Embedding(vocab_sizes['income'], emb_dim)

        input_dim = emb_dim + emb_dim // 2 + emb_dim + 1

        self.fc_mu = nn.Linear(input_dim, latent_dim)
        self.fc_logvar = nn.Linear(input_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim),
        )

        self.out_age = nn.Linear(emb_dim, vocab_sizes['age'])
        self.out_sex = nn.Linear(emb_dim // 2, vocab_sizes['sex'])
        self.out_income = nn.Linear(emb_dim, vocab_sizes['income'])
        self.out_kids = nn.Linear(1, 1)

        self.emb_dim = emb_dim

    def encode(self, u_age, u_sex, u_income, u_kids):
        x = torch.cat([
            self.emb_age(u_age),
            self.emb_sex(u_sex),
            self.emb_income(u_income),
            u_kids.unsqueeze(1),
        ], dim=1)
        return self.fc_mu(x), self.fc_logvar(x)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        d = self.decoder(z)
        off = 0
        d_age = d[:, off:off + self.emb_dim]; off += self.emb_dim
        d_sex = d[:, off:off + self.emb_dim // 2]; off += self.emb_dim // 2
        d_income = d[:, off:off + self.emb_dim]; off += self.emb_dim
        d_kids = d[:, off:off + 1]
        return (
            self.out_age(d_age),
            self.out_sex(d_sex),
            self.out_income(d_income),
            self.out_kids(d_kids).squeeze(1),
        )

    def forward(self, u_age, u_sex, u_income, u_kids):
        mu, logvar = self.encode(u_age, u_sex, u_income, u_kids)
        z = self.reparameterize(mu, logvar)
        return z, self.decode(z), mu, logvar

In [11]:
class ItemVAEEncoder(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=64, latent_dim=8):
        super().__init__()
        self.emb_type = nn.Embedding(vocab_sizes['content_type'], emb_dim)
        self.emb_decade = nn.Embedding(vocab_sizes['release_decade'], emb_dim)

        input_dim = emb_dim * 2 + 1

        self.fc_mu = nn.Linear(input_dim, latent_dim)
        self.fc_logvar = nn.Linear(input_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim),
        )

        self.out_type = nn.Linear(emb_dim, vocab_sizes['content_type'])
        self.out_decade = nn.Linear(emb_dim, vocab_sizes['release_decade'])
        self.out_age_rating = nn.Linear(1, 1)

        self.emb_dim = emb_dim

    def encode(self, i_type, i_decade, i_age_rating):
        x = torch.cat([
            self.emb_type(i_type),
            self.emb_decade(i_decade),
            i_age_rating.unsqueeze(1),
        ], dim=1)
        return self.fc_mu(x), self.fc_logvar(x)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        d = self.decoder(z)
        off = 0
        d_type = d[:, off:off + self.emb_dim]; off += self.emb_dim
        d_decade = d[:, off:off + self.emb_dim]; off += self.emb_dim
        d_age = d[:, off:off + 1]
        return (
            self.out_type(d_type),
            self.out_decade(d_decade),
            self.out_age_rating(d_age).squeeze(1),
        )

    def forward(self, i_type, i_decade, i_age_rating):
        mu, logvar = self.encode(i_type, i_decade, i_age_rating)
        z = self.reparameterize(mu, logvar)
        return z, self.decode(z), mu, logvar

In [12]:
class HybridVAE(nn.Module):
    def __init__(self, vocab_sizes, vocabularies, emb_dim=16, hidden_dim=64, latent_dim=8):
        super().__init__()
        self.user_vae = UserVAEEncoder(vocab_sizes, emb_dim, hidden_dim, latent_dim)
        self.item_vae = ItemVAEEncoder(vocab_sizes, emb_dim, hidden_dim, latent_dim)
        self.denoiser = DenoisingWrapper(noise_prob=0.15)
        self._vocabularies = vocabularies

    def forward(self, batch, training=True):
        if training:
            u_age = self.denoiser.add_noise(batch['u_age'], len(self._vocabularies['age']))
            u_sex = self.denoiser.add_noise(batch['u_sex'], len(self._vocabularies['sex']))
            u_income = self.denoiser.add_noise(batch['u_income'], len(self._vocabularies['income']))
            i_type = self.denoiser.add_noise(batch['i_type'], len(self._vocabularies['content_type']))
            i_decade = self.denoiser.add_noise(batch['i_decade'], len(self._vocabularies['release_decade']))
        else:
            u_age, u_sex, u_income = batch['u_age'], batch['u_sex'], batch['u_income']
            i_type, i_decade = batch['i_type'], batch['i_decade']

        u_z, u_recon, u_mu, u_logvar = self.user_vae(u_age, u_sex, u_income, batch['u_kids'])
        i_z, i_recon, i_mu, i_logvar = self.item_vae(i_type, i_decade, batch['i_age_rating'])

        return u_z, i_z, u_recon, i_recon, u_mu, u_logvar, i_mu, i_logvar

In [13]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VAE_CONFIG = {'emb_dim': 16, 'hidden_dim': 64, 'latent_dim': 8}

vae_model = HybridVAE(vocab_sizes=vocab_sizes, vocabularies=vocabularies, **VAE_CONFIG).to(DEVICE)
print(f'params: {sum(p.numel() for p in vae_model.parameters()):,}')

params: 8,025


## Loss functions

In [14]:
def vae_reconstruction_loss(u_recon, i_recon, batch):
    u_age, u_sex, u_income, u_kids = u_recon
    i_type, i_decade, i_age = i_recon

    loss = (
        F.cross_entropy(u_age, batch['u_age'])
        + F.cross_entropy(u_sex, batch['u_sex'])
        + F.cross_entropy(u_income, batch['u_income'])
        + F.binary_cross_entropy_with_logits(u_kids, batch['u_kids'])
        + F.cross_entropy(i_type, batch['i_type'])
        + F.cross_entropy(i_decade, batch['i_decade'])
        + F.mse_loss(i_age, batch['i_age_rating'])
    )
    return loss

In [15]:
def kl_divergence(mu, logvar):
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

## VAE training

In [16]:
NUM_EPOCHS = 15
LR = 5e-3
KL_WEIGHT = 0.1

optimizer = optim.AdamW(vae_model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

In [17]:
train_losses, recon_losses, kl_losses = [], [], []

for epoch in range(NUM_EPOCHS):
    vae_model.train()
    ep_total = ep_recon = ep_kl = 0.0
    start = time.time()

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        u_z, i_z, u_recon, i_recon, u_mu, u_logvar, i_mu, i_logvar = vae_model(batch, training=True)

        recon = vae_reconstruction_loss(u_recon, i_recon, batch)
        kl = kl_divergence(u_mu, u_logvar) + kl_divergence(i_mu, i_logvar)
        loss = recon + KL_WEIGHT * kl

        loss.backward()
        torch.nn.utils.clip_grad_norm_(vae_model.parameters(), max_norm=1.0)
        optimizer.step()

        ep_total += loss.item()
        ep_recon += recon.item()
        ep_kl += kl.item()

    scheduler.step()

    n = len(train_loader)
    train_losses.append(ep_total / n)
    recon_losses.append(ep_recon / n)
    kl_losses.append(ep_kl / n)

    print(f'epoch {epoch + 1}/{NUM_EPOCHS} | total {train_losses[-1]:.4f} | recon {recon_losses[-1]:.4f} | kl {kl_losses[-1]:.4f} | {time.time() - start:.1f}s')

epoch 1/15 | total 7.8800 | recon 7.6831 | kl 1.9688 | 65.4s
epoch 2/15 | total 7.6651 | recon 7.6604 | kl 0.0476 | 64.2s
epoch 3/15 | total 7.6616 | recon 7.6599 | kl 0.0169 | 63.7s
epoch 4/15 | total 7.6607 | recon 7.6598 | kl 0.0087 | 63.6s
epoch 5/15 | total 7.6601 | recon 7.6597 | kl 0.0037 | 64.2s
epoch 6/15 | total 7.6598 | recon 7.6596 | kl 0.0017 | 64.0s
epoch 7/15 | total 7.6596 | recon 7.6595 | kl 0.0011 | 63.9s
epoch 8/15 | total 7.6595 | recon 7.6594 | kl 0.0008 | 63.9s
epoch 9/15 | total 7.6593 | recon 7.6593 | kl 0.0004 | 63.8s
epoch 10/15 | total 7.6592 | recon 7.6592 | kl 0.0002 | 64.2s
epoch 11/15 | total 7.6591 | recon 7.6591 | kl 0.0001 | 64.2s
epoch 12/15 | total 7.6590 | recon 7.6590 | kl 0.0001 | 64.2s
epoch 13/15 | total 7.6590 | recon 7.6590 | kl 0.0000 | 64.3s
epoch 14/15 | total 7.6589 | recon 7.6589 | kl 0.0000 | 64.3s
epoch 15/15 | total 7.6589 | recon 7.6589 | kl 0.0000 | 64.0s


## Extract embeddings

In [18]:
def extract_user_embeddings(model, users_features, vocabularies, device, batch_size=1024):
    model.eval()
    embs, ids = [], []
    for i in tqdm(range(0, len(users_features), batch_size)):
        df = users_features.iloc[i:i + batch_size]
        n = len(df)
        batch = {
            'u_age': torch.tensor(df['age'].fillna('Unknown').astype(str).map(vocabularies['age']).fillna(0).values, dtype=torch.long, device=device),
            'u_sex': torch.tensor(df['sex'].fillna('Unknown').astype(str).map(vocabularies['sex']).fillna(0).values, dtype=torch.long, device=device),
            'u_income': torch.tensor(df['income'].fillna('Unknown').astype(str).map(vocabularies['income']).fillna(0).values, dtype=torch.long, device=device),
            'u_kids': torch.tensor(df['kids_flg'].fillna(0).values, dtype=torch.float32, device=device),
            'i_type': torch.zeros(n, dtype=torch.long, device=device),
            'i_decade': torch.zeros(n, dtype=torch.long, device=device),
            'i_age_rating': torch.zeros(n, device=device),
        }
        with torch.no_grad():
            u_z, _, _, _, _, _, _, _ = model(batch, training=False)
        embs.append(u_z.cpu().numpy())
        ids.extend(df['user_id'].values)
    return np.vstack(embs), np.array(ids)

In [19]:
def extract_item_embeddings(model, items_features, vocabularies, device, batch_size=1024):
    model.eval()
    embs, ids = [], []
    for i in tqdm(range(0, len(items_features), batch_size)):
        df = items_features.iloc[i:i + batch_size]
        n = len(df)
        batch = {
            'i_type': torch.tensor(df['content_type'].fillna('Unknown').astype(str).map(vocabularies['content_type']).fillna(0).values, dtype=torch.long, device=device),
            'i_decade': torch.tensor(df['release_decade'].fillna('Unknown').astype(str).map(vocabularies['release_decade']).fillna(0).values, dtype=torch.long, device=device),
            'i_age_rating': torch.tensor(df['age_rating_ord'].fillna(2).values, dtype=torch.float32, device=device),
            'u_age': torch.zeros(n, dtype=torch.long, device=device),
            'u_sex': torch.zeros(n, dtype=torch.long, device=device),
            'u_income': torch.zeros(n, dtype=torch.long, device=device),
            'u_kids': torch.zeros(n, device=device),
        }
        with torch.no_grad():
            _, i_z, _, _, _, _, _, _ = model(batch, training=False)
        embs.append(i_z.cpu().numpy())
        ids.extend(df['item_id'].values)
    return np.vstack(embs), np.array(ids)

In [20]:
user_embs, user_emb_ids = extract_user_embeddings(vae_model, users_features, vocabularies, DEVICE)
item_embs, item_emb_ids = extract_item_embeddings(vae_model, items_features, vocabularies, DEVICE)

user_emb_df = pd.DataFrame(user_embs, columns=[f'user_emb_{i}' for i in range(user_embs.shape[1])])
user_emb_df['user_id'] = user_emb_ids.astype(np.int64)

item_emb_df = pd.DataFrame(item_embs, columns=[f'item_emb_{i}' for i in range(item_embs.shape[1])])
item_emb_df['item_id'] = item_emb_ids.astype(np.int64)

print(f'user_embs: {user_embs.shape}, item_embs: {item_embs.shape}')

100%|██████████| 16/16 [00:00<00:00, 298.05it/s]

user_embs: (840197, 8), item_embs: (15963, 8)


In [21]:
users_with_emb = users_features.merge(user_emb_df, on='user_id', how='left')
items_with_emb = items_features.merge(item_emb_df, on='item_id', how='left')

print(f'users_with_emb: {users_with_emb.shape}')
print(f'items_with_emb: {items_with_emb.shape}')

users_with_emb: (840197, 17)
items_with_emb: (15963, 20)


## CatBoost feature schema

In [22]:
CAT_FEATURES = ['age', 'sex', 'income', 'content_type', 'release_decade']
USER_NUM = ['kids_flg', 'user_total_interactions_log', 'user_avg_duration_log']
ITEM_NUM = ['age_rating_ord', 'is_recent_release', 'is_kids_genre']
USER_EMB = [f'user_emb_{i}' for i in range(user_embs.shape[1])]
ITEM_EMB = [f'item_emb_{i}' for i in range(item_embs.shape[1])]
CROSS = ['is_kids_user_and_kids_content']

NUM_FEATURES = USER_NUM + ITEM_NUM + USER_EMB + ITEM_EMB + CROSS
FEATURES = CAT_FEATURES + NUM_FEATURES

print(f'features ({len(FEATURES)}): {FEATURES}')

features (28): ['age', 'sex', 'income', 'content_type', 'release_decade', 'kids_flg', 'user_total_interactions_log', 'user_avg_duration_log', 'age_rating_ord', 'is_recent_release', 'is_kids_genre', 'user_emb_0', 'user_emb_1', 'user_emb_2', 'user_emb_3', 'user_emb_4', 'user_emb_5', 'user_emb_6', 'user_emb_7', 'item_emb_0', 'item_emb_1', 'item_emb_2', 'item_emb_3', 'item_emb_4', 'item_emb_5', 'item_emb_6', 'item_emb_7', 'is_kids_user_and_kids_content']


## Assemble training data

In [23]:
def build_features(df, users_with_emb, items_with_emb):
    out = df[['user_id', 'item_id', 'target']].merge(users_with_emb, on='user_id', how='left')
    out = out.merge(items_with_emb, on='item_id', how='left')
    out['is_kids_user_and_kids_content'] = (
        (out['kids_flg'] == 1) & (out['age_rating_ord'] <= 1)
    ).astype('int8')
    for c in CAT_FEATURES:
        out[c] = out[c].fillna('Unknown').astype(str)
    for c in NUM_FEATURES:
        out[c] = out[c].fillna(0.0).astype('float32')
    return out

train_cb = build_features(train_final, users_with_emb, items_with_emb)
print(f'train_cb: {train_cb.shape}')

train_cb: (9178632, 39)


## CatBoost training

In [24]:
X = train_cb[FEATURES]
y = train_cb['target']

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

del X, y
gc.collect()

10

In [25]:
cb_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='Logloss',
    random_seed=42,
    verbose=100,
    cat_features=CAT_FEATURES,
    task_type='GPU',
    devices='0',
    early_stopping_rounds=50,
    use_best_model=True,
)

cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

0:	learn: 0.6662481	test: 0.6662617	best: 0.6662617 (0)	total: 851ms	remaining: 7m 4s
100:	learn: 0.3762618	test: 0.3764149	best: 0.3764149 (100)	total: 1m 10s	remaining: 4m 37s
200:	learn: 0.3443826	test: 0.3446754	best: 0.3446754 (200)	total: 2m 17s	remaining: 3m 25s
300:	learn: 0.3247930	test: 0.3250773	best: 0.3250773 (300)	total: 3m 24s	remaining: 2m 15s
400:	learn: 0.3099101	test: 0.3102511	best: 0.3102511 (400)	total: 4m 30s	remaining: 1m 6s
499:	learn: 0.2984375	test: 0.2988289	best: 0.2988289 (499)	total: 5m 36s	remaining: 0us
bestTest = 0.298828891
bestIteration = 499


CatBoostClassifier(cat_features=['age', 'sex', 'income', 'content_type', 'release_decade'], depth=6, devices='0', early_stopping_rounds=50, eval_metric='Logloss', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', use_best_model=True, verbose=100)

In [26]:
del X_tr, X_val, y_tr, y_val, train_cb
gc.collect()

print(f'Best iteration: {cb_model.get_best_iteration()}')
print(f'Best Logloss: {cb_model.get_best_score()["validation"]["Logloss"]:.4f}')

Best iteration: 499
Best Logloss: 0.2988


## Metrics

In [27]:
def compute_metrics(top_k_items, true_items, k=10):
    true_set = set(true_items)
    hits = 0
    sum_precision = 0.0
    sum_ndcg = 0.0
    for rank, item in enumerate(top_k_items[:k], start=1):
        if item in true_set:
            hits += 1
            sum_precision += hits / rank
            sum_ndcg += 1.0 / np.log2(rank + 1)
    num_relevant = min(k, len(true_set))
    if num_relevant == 0:
        return 0.0, 0.0, 0.0
    map_ = sum_precision / num_relevant
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, num_relevant + 1))
    ndcg = sum_ndcg / ideal_dcg if ideal_dcg > 0 else 0.0
    hr = 1.0 if hits > 0 else 0.0
    return hr, ndcg, map_

In [28]:
def evaluate(sampled_users, candidates, ground_truth, users_with_emb, items_with_emb, batch_size=20, k=10):
    hr_list, ndcg_list, map_list = [], [], []
    for i in tqdm(range(0, len(sampled_users), batch_size)):
        batch = sampled_users[i:i + batch_size]
        rows = [{'user_id': int(u), 'item_id': iid} for u in batch for iid in candidates[int(u)]]
        df = pd.DataFrame(rows)
        df = df.merge(users_with_emb, on='user_id', how='left')
        df = df.merge(items_with_emb, on='item_id', how='left')
        df['is_kids_user_and_kids_content'] = ((df['kids_flg'] == 1) & (df['age_rating_ord'] <= 1)).astype('int8')
        for c in CAT_FEATURES:
            df[c] = df[c].fillna('Unknown').astype(str)
        for c in NUM_FEATURES:
            df[c] = df[c].fillna(0.0).astype('float32')

        df['score'] = cb_model.predict_proba(df[FEATURES])[:, 1]

        for u in batch:
            u = int(u)
            preds = df[df['user_id'] == u].sort_values('score', ascending=False)
            hr, ndcg, map_ = compute_metrics(preds['item_id'].head(k).tolist(), ground_truth.get(u, []), k=k)
            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(map_)
    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))

## Evaluate on train sample

In [29]:
hr_train, ndcg_train, map_train = evaluate(
    eval_data['train_sampled_users'],
    eval_data['train_candidates'],
    eval_data['train_ground_truth'],
    users_with_emb,
    items_with_emb,
)

100%|██████████| 25/25 [00:05<00:00,  4.60it/s]


## Evaluate on val sample

In [30]:
hr_val, ndcg_val, map_val = evaluate(
    eval_data['val_sampled_users'],
    eval_data['val_candidates'],
    eval_data['val_ground_truth'],
    users_with_emb,
    items_with_emb,
)

100%|██████████| 25/25 [00:05<00:00,  4.58it/s]


## Results

In [31]:
results = pd.DataFrame({
    'Metric': ['HR@10', 'NDCG@10', 'MAP@10'],
    'Train': [hr_train, ndcg_train, map_train],
    'Val': [hr_val, ndcg_val, map_val],
})
results['Gap'] = results['Train'] - results['Val']
print(results.to_string(index=False))

 Metric    Train      Val       Gap
  HR@10 0.418000 0.452000 -0.034000
NDCG@10 0.199473 0.176230  0.023243
 MAP@10 0.139727 0.112412  0.027315


## Save artifacts

In [32]:
torch.save({
    'state_dict': vae_model.state_dict(),
    'vocabularies': vocabularies,
    'vocab_sizes': vocab_sizes,
    'config': VAE_CONFIG,
}, f'{OUTPUT_DIR}vae_model.pt')

cb_model.save_model(f'{OUTPUT_DIR}catboost_vae.cbm')

user_emb_df.to_parquet(f'{OUTPUT_DIR}users_emb.parquet', index=False)
item_emb_df.to_parquet(f'{OUTPUT_DIR}items_emb.parquet', index=False)

print('artifacts saved: vae_model.pt, catboost_vae.cbm, users_emb.parquet, items_emb.parquet')

artifacts saved: vae_model.pt, catboost_vae.cbm, users_emb.parquet, items_emb.parquet
